# Headroom test — does Qwen3-4B actually know more than Qwen3-0.6B on GSM8K?

**Purpose.** Before spending time on the on-policy vs. off-policy distillation
experiment, we need to check the premise it depends on: that the teacher
(Qwen3-4B) is meaningfully better than the student (Qwen3-0.6B) at the task
we are distilling. If the two models are already close on GSM8K, there is
little for the student to learn, regardless of which distillation method
we use — and we would have to change something (the models/dataset used).

**What this notebook does.** It evaluates both models zero-shot (no
training, no distillation) on the same random subset of the GSM8K test set,
in the same way that we will do in our actual experiment. It then reports:
- accuracy for each model, with a 95% confidence interval,
- the accuracy gap between the two models, with its own confidence
  interval computed on the same questions (a paired comparison, which is
  more precise than comparing two separate intervals),
- mean response length and how often generation hit the token limit
  (a high truncation rate would mean answers are being cut off, which
  would make a model look worse than it is).

**Decision rule**:
- **Gap ≥ ~10 percentage points, and the gap's confidence interval does not
  include 0:** there is headroom — proceed with the distillation plan
  on this task.
- **Gap < ~5 points, or its confidence interval includes 0:** there is
  little to no headroom on this task as currently set up - switch to a
  harder dataset (e.g. MATH levels 1-3) or a different teacher/student pair
  before running the main experiment.
- **In between:** proceed, but expect small effects and plan for more seeds.

**Setup:** Accelerator = GPU T4 x2 (only 1 GPU is actually needed - models are read-only), Internet = on.

In [1]:
!pip install -q -U transformers datasets accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import gc
import csv
import json
import math
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Transformers:", __import__("transformers").__version__)
print("Datasets:", __import__("datasets").__version__)
print("PyTorch:", torch.__version__)

assert torch.cuda.is_available(), "No GPU visible — check the accelerator setting."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {round(p.total_memory / 1e9, 1)} GB, compute capability {p.major}.{p.minor}")

Transformers: 5.22.1
Datasets: 5.1.0
PyTorch: 2.11.0+cu130
GPU 0: Tesla T4, 15.6 GB, compute capability 7.5
GPU 1: Tesla T4, 15.6 GB, compute capability 7.5


## Config

In [3]:
TEACHER = "Qwen/Qwen3-4B"
STUDENT = "Qwen/Qwen3-0.6B"

# This script loads one model at a time (see evaluate_model below), so only
# one GPU is ever used at once. cuda:0 is used whether the session has one
# T4 or two.
DEVICE = "cuda:0"

N_EVAL = 300            # number of GSM8K test problems to evaluate (of 1319 total)
BATCH_SIZE = 16         # prompts generated together; halves automatically on out-of-memory
MAX_NEW_TOKENS = 384    # generation length cap per answer, matches the distillation notebook

# Same prompt as the rest of the project, so this test measures the same
# thing the distillation experiment will measure.
INSTRUCTION = "Solve the problem step by step. Put the final number at the end."
END_TOKEN = "<|im_end|>"   # Qwen chat end-of-turn token
SEED = 42

# T4 (compute capability 7.5) has no native bf16, so this falls back to fp16;
# a newer GPU would use bf16 instead.
HALF = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("half precision dtype:", HALF)

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

half precision dtype: torch.float16


## Helpers

Grading and prompt-building are copied from the rest of the project so this
test measures accuracy the same way the main experiment will.

In [4]:
def bytes_to_gb(num_bytes):
    """Convert a byte count to gigabytes, rounded for readable printing."""
    return round(num_bytes / 1024**3, 2)


def vram_used_gb(device="cuda:0"):
    """Whole-GPU memory currently in use on one device (same as nvidia-smi).

    This is not limited to memory PyTorch itself has allocated, so it is
    the right number to check when confirming a model actually fits.

    Args:
        device: CUDA device string, e.g. "cuda:0".

    Returns:
        float: GPU memory in use, in GB.
    """
    free, total = torch.cuda.mem_get_info(device)
    return bytes_to_gb(total - free)


def extract_last_number(text):
    """Return the last number in a piece of text as a float, or None.

    This is the model's predicted answer: GSM8K answers are graded by the
    final number stated, so we take whatever number the model wrote last.
    """
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(numbers[-1]) if numbers else None


def gold_answer(answer_field):
    """Extract the correct number from a GSM8K reference solution.

    GSM8K answers end with '#### <number>'; this pulls out that number so
    it can be compared against the model's prediction.
    """
    return extract_last_number(answer_field.split("####")[-1])


# Loaded once below, after the sanity check confirms teacher and student
# share a tokenizer. Declared here (as a global) so prompt_ids() and
# eos_token_ids() below can use it without passing it through every call —
# the same pattern the distillation notebook uses.
tokenizer = None


def prompt_ids(question):
    """Build the chat prompt (question + instruction) as a list of token ids.

    Thinking mode is off, matching the rest of the project, so responses
    stay short and comparable across models.

    Args:
        question: The GSM8K question text.

    Returns:
        list[int]: Token ids for the full prompt, ready for generation.
    """
    messages = [{"role": "user", "content": f"{question}\n\n{INSTRUCTION}"}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def eos_token_ids(model):
    """Collect every token id that marks the end of this model's response.

    Combines the model's own configured end-of-sequence id(s) with Qwen's
    chat end-of-turn token, since either can mark where an answer finishes.

    Args:
        model: A loaded causal LM; only its generation_config is used.

    Returns:
        set[int]: Token ids that should be treated as "answer finished".
    """
    eos = model.generation_config.eos_token_id
    ids = set(eos if isinstance(eos, list) else [eos])
    ids.add(tokenizer.convert_tokens_to_ids(END_TOKEN))
    return ids

## Tokenizer, and a vocabulary sanity check

Both this notebook and the distillation notebook assume teacher and student
use the *same* tokenizer. If that were false, token ids produced by one
model would not mean the same thing to the other, and nothing downstream
would be comparable. This is cheap to check directly, so we do.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT)

# Load only the teacher's tokenizer here (not the 4B model itself) to keep
# this check cheap; the full teacher model is loaded later, inside
# evaluate_model, only when it's actually needed.
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER)
same_vocab = teacher_tokenizer.get_vocab() == tokenizer.get_vocab()
print(f"Teacher and student share an identical vocabulary: {same_vocab}")
assert same_vocab, "Teacher/student tokenizers differ -- results below would not be comparable."
del teacher_tokenizer

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

Teacher and student share an identical vocabulary: True


## Data

A fixed-seed **random** subset of the test set is used (rather than the
first `N_EVAL` problems), so the headroom estimate is not biased by GSM8K's
problem ordering. The seed keeps the subset the same across reruns, so
results are comparable.

To reuse this notebook for a different dataset (e.g. MATH), replace the
`load_dataset` call and `gold_answer` with that dataset's loading and
grading logic; everything else is dataset-agnostic.

In [6]:
full_test = load_dataset("openai/gsm8k", "main", split="test")
print(f"GSM8K test set has {len(full_test)} problems")

rng = random.Random(SEED)
indices = list(range(len(full_test)))
rng.shuffle(indices)
eval_indices = indices[:N_EVAL]
problems = [full_test[i] for i in eval_indices]
print(f"Evaluating on a random, seed-fixed subset of {len(problems)} problems")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test set has 1319 problems
Evaluating on a random, seed-fixed subset of 300 problems


## Batched generation

Generates answers for several prompts at once instead of one at a time,
which is the main speed-up over the project's earlier serial evaluation
script. Falls back to smaller batches on out-of-memory instead of crashing
the whole run.

In [7]:
@torch.no_grad()
def generate_batch(model, device, prompts, eos_ids, max_new_tokens, dtype):
    """Generate greedy, deterministic responses for a batch of prompts.

    Prompts are left-padded to a common width so the GPU can process them
    as one rectangular tensor; a model only writes new tokens after its own
    prompt ends, so each real prompt has to sit at the right edge of the
    padded block (left padding), with the attention mask marking which
    positions are real tokens vs. padding.

    Decoding is greedy (do_sample=False), not sampled, so that reruns of
    this notebook give identical answers -- important for a comparison the
    team will rely on.

    If a batch does not fit in memory, it is split in half and retried;
    this degrades gracefully toward batch size 1 instead of aborting the
    whole evaluation run on one bad batch.

    Args:
        model: The model to generate from (already on `device`, eval mode).
        device: CUDA device string the model lives on.
        prompts: List of prompts, each already tokenized to a list of ids.
        eos_ids: Token ids marking the end of an answer; used here only to
            trim trailing padding after generation finishes.
        max_new_tokens: Maximum number of new tokens to generate per prompt.
        dtype: torch dtype to run generation in under autocast (fp16/bf16).

    Returns:
        list[list[int]]: One generated token-id sequence per prompt, in the
        same order as `prompts`, with padding after the first end token
        removed.
    """
    if len(prompts) == 0:
        return []

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    width = max(len(p) for p in prompts)
    input_ids = torch.full((len(prompts), width), pad_id, dtype=torch.long, device=device)
    attention_mask = torch.zeros_like(input_ids)
    for i, p in enumerate(prompts):
        input_ids[i, width - len(p):] = torch.tensor(p, device=device)
        attention_mask[i, width - len(p):] = 1   # 1 = real token, 0 = padding to ignore

    try:
        with torch.autocast("cuda", dtype=dtype):
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy: deterministic, repeatable comparison
                pad_token_id=pad_id,
                use_cache=True,
            )
    except torch.cuda.OutOfMemoryError:
        if len(prompts) == 1:
            raise  # nothing smaller to fall back to; a real problem
        # Safety net: this batch doesn't fit, so split it and retry rather
        # than aborting the whole evaluation run.
        gc.collect()
        torch.cuda.empty_cache()
        mid = len(prompts) // 2
        return (generate_batch(model, device, prompts[:mid], eos_ids, max_new_tokens, dtype)
                + generate_batch(model, device, prompts[mid:], eos_ids, max_new_tokens, dtype))

    # Trim the padding from each answer: keep up to and including the first
    # end token; if none appears, the whole row is kept (it hit the cap).
    responses = []
    for row in out[:, width:].tolist():
        end = len(row)
        for j, tok in enumerate(row):
            if tok in eos_ids:
                end = j + 1
                break
        responses.append(row[:end])
    return responses

## Statistics

Two different intervals are computed, because they answer different
questions:

- **Wilson score interval** — a confidence interval for *one* model's
  accuracy on its own. Used to sanity-check that each model's accuracy is
  reasonable (not so close to 0% or 100% that there's nothing to measure).
- **Paired bootstrap interval** — a confidence interval for the *gap*
  between the two models, computed properly by resampling questions (not
  each model's correctness independently). Because both models answered
  the exact same questions, pairing removes noise that comes from
  question-to-question difficulty, and gives a tighter, more honest
  interval than just comparing two separate Wilson intervals. **This is
  the number the decision rule is based on.**

In [8]:
def wilson_ci(successes, n, z=1.96):
    """Wilson score confidence interval for a binomial proportion.

    Used instead of the simpler normal-approximation interval because it
    stays inside [0, 1] and stays reasonably accurate even for small sample
    sizes or accuracies close to 0 or 1 -- both plausible here.

    Args:
        successes: Number of correct answers.
        n: Total number of questions evaluated.
        z: Standard-normal critical value; 1.96 corresponds to 95% confidence.

    Returns:
        tuple[float, float]: (lower, upper) bound of the confidence interval.
    """
    if n == 0:
        return (0.0, 0.0)
    p = successes / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - margin) / denom, (centre + margin) / denom)


def paired_bootstrap_diff_ci(correct_a, correct_b, n_boot=10000, seed=SEED):
    """Bootstrap confidence interval for the accuracy gap between two models.

    correct_a and correct_b must be 0/1 outcomes on the SAME questions in
    the SAME order (they are paired, since both models were evaluated on
    identical problems here). Each bootstrap resample draws QUESTIONS with
    replacement -- not each model's correctness independently -- which
    keeps an easy or hard question's effect on both models linked together
    in every resample, giving a tighter and more honest interval than
    treating the two accuracies as independent.

    Args:
        correct_a: 0/1 list, model A's correctness per question.
        correct_b: 0/1 list, model B's correctness, same question order.
        n_boot: Number of bootstrap resamples.
        seed: RNG seed, so the interval is reproducible on rerun.

    Returns:
        tuple[float, tuple[float, float]]: (observed accuracy gap, (lower,
        upper) 95% interval for that gap), as accuracy of A minus accuracy
        of B.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(correct_a, dtype=float)
    b = np.asarray(correct_b, dtype=float)
    n = len(a)
    assert n == len(b), "correct_a and correct_b must be the same length (paired)"

    observed_diff = a.mean() - b.mean()
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)   # resample QUESTIONS, not each model separately
        diffs[i] = a[idx].mean() - b[idx].mean()
    lower, upper = np.percentile(diffs, [2.5, 97.5])
    return observed_diff, (lower, upper)

## Evaluation routine

Loads one model, evaluates it on every problem in batches, frees it from
memory, and returns a summary plus per-question records. Records are kept
(not just the final accuracy number) so wrong answers can be read back
later rather than trusting a single number blindly.

In [9]:
def evaluate_model(model_name, device, problems, batch_size, max_new_tokens, dtype):
    """Run a full zero-shot, greedy evaluation of one model on a problem set.

    The model is loaded fresh, evaluated, then deleted before returning --
    so teacher and student are never required to be in memory at the same
    time. That is the reason this notebook needs only one GPU: unlike
    training, a zero-shot check needs no gradients or optimizer state for
    either model, and the two models are never used simultaneously.

    Args:
        model_name: Hugging Face model id (e.g. "Qwen/Qwen3-4B").
        device: CUDA device string to load this model onto.
        problems: List of GSM8K examples (dicts with "question" and
            "answer", as returned by the datasets library).
        batch_size: Prompts generated together per call; generate_batch
            automatically falls back to smaller batches on out-of-memory.
        max_new_tokens: Generation length cap per answer.
        dtype: torch dtype for autocast during generation.

    Returns:
        dict: {
            "model": model_name,
            "n": number of problems evaluated,
            "correct": number of correct answers,
            "accuracy": correct / n,
            "ci_95": Wilson 95% confidence interval for accuracy,
            "mean_response_tokens": average generated length,
            "truncated_fraction": fraction of answers that never produced
                an end token before hitting max_new_tokens (a high value
                here means answers may be getting cut off mid-reasoning,
                which would understate this model's real accuracy),
            "elapsed_seconds": total generation + grading wall-clock time,
            "records": one dict per question, for later inspection,
        }
    """
    print(f"\n{'=' * 60}\nLoading {model_name} onto {device}\n{'=' * 60}")
    t_load = time.time()
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)   # inference only: never build a graph for this model
    print(f"Loaded in {time.time() - t_load:.1f}s | VRAM on {device}: {vram_used_gb(device)} GB")

    eos_ids = eos_token_ids(model)
    records = []
    t_start = time.time()

    for batch_start in range(0, len(problems), batch_size):
        batch = problems[batch_start: batch_start + batch_size]
        prompts = [prompt_ids(ex["question"]) for ex in batch]
        responses = generate_batch(model, device, prompts, eos_ids, max_new_tokens, dtype)

        for ex, resp_ids in zip(batch, responses):
            response_text = tokenizer.decode(resp_ids, skip_special_tokens=True)
            predicted = extract_last_number(response_text)
            expected = gold_answer(ex["answer"])
            is_correct = predicted is not None and abs(predicted - expected) < 1e-6
            # No end token anywhere in the kept response means generation
            # hit max_new_tokens without the model finishing on its own.
            truncated = (len(resp_ids) == 0) or (resp_ids[-1] not in eos_ids)
            records.append({
                "question": ex["question"],
                "response": response_text,
                "predicted": predicted,
                "expected": expected,
                "correct": is_correct,
                "response_tokens": len(resp_ids),
                "truncated": truncated,
            })

        done = len(records)
        elapsed = time.time() - t_start
        rate = elapsed / done if done else 0.0
        remaining = rate * (len(problems) - done)
        running_acc = sum(r["correct"] for r in records) / done
        print(f"  {done}/{len(problems)} done | {elapsed:.0f}s elapsed | "
              f"~{remaining:.0f}s remaining | running accuracy {running_acc:.3f}")

    elapsed_total = time.time() - t_start
    correct = sum(r["correct"] for r in records)
    n = len(records)
    accuracy = correct / n
    ci_low, ci_high = wilson_ci(correct, n)
    truncated_fraction = sum(r["truncated"] for r in records) / n
    mean_len = sum(r["response_tokens"] for r in records) / n

    summary = {
        "model": model_name,
        "n": n,
        "correct": correct,
        "accuracy": round(accuracy, 4),
        "ci_95": (round(ci_low, 4), round(ci_high, 4)),
        "mean_response_tokens": round(mean_len, 1),
        "truncated_fraction": round(truncated_fraction, 4),
        "elapsed_seconds": round(elapsed_total, 1),
        "records": records,
    }

    print(f"\n{model_name}: accuracy {accuracy:.3f} "
          f"(95% CI {ci_low:.3f}-{ci_high:.3f}) | "
          f"mean length {mean_len:.1f} tokens | "
          f"truncated {truncated_fraction:.1%} | "
          f"{elapsed_total:.0f}s total")

    # Free this model before the next one loads. Without this, both models
    # would need to fit in memory at once, which this zero-shot check never
    # actually requires -- it's why the script can run on a single GPU.
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return summary

## Run both models

Teacher first, then student. Each is loaded, evaluated on the identical
problem set, and freed before the next one loads.

In [10]:
teacher_summary = evaluate_model(TEACHER, DEVICE, problems, BATCH_SIZE, MAX_NEW_TOKENS, HALF)


Loading Qwen/Qwen3-4B onto cuda:0


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded in 58.4s | VRAM on cuda:0: 7.65 GB
  16/300 done | 37s elapsed | ~655s remaining | running accuracy 0.875
  32/300 done | 75s elapsed | ~625s remaining | running accuracy 0.875
  48/300 done | 117s elapsed | ~614s remaining | running accuracy 0.771
  64/300 done | 155s elapsed | ~572s remaining | running accuracy 0.797
  80/300 done | 190s elapsed | ~522s remaining | running accuracy 0.825
  96/300 done | 227s elapsed | ~483s remaining | running accuracy 0.823
  112/300 done | 267s elapsed | ~448s remaining | running accuracy 0.821
  128/300 done | 307s elapsed | ~413s remaining | running accuracy 0.828
  144/300 done | 347s elapsed | ~376s remaining | running accuracy 0.819
  160/300 done | 385s elapsed | ~336s remaining | running accuracy 0.825
  176/300 done | 422s elapsed | ~297s remaining | running accuracy 0.835
  192/300 done | 463s elapsed | ~261s remaining | running accuracy 0.844
  208/300 done | 501s elapsed | ~222s remaining | running accuracy 0.841
  224/300 done | 

In [11]:
student_summary = evaluate_model(STUDENT, DEVICE, problems, BATCH_SIZE, MAX_NEW_TOKENS, HALF)


Loading Qwen/Qwen3-0.6B onto cuda:0


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded in 13.1s | VRAM on cuda:0: 1.26 GB
  16/300 done | 18s elapsed | ~311s remaining | running accuracy 0.812
  32/300 done | 35s elapsed | ~293s remaining | running accuracy 0.656
  48/300 done | 53s elapsed | ~277s remaining | running accuracy 0.604
  64/300 done | 70s elapsed | ~259s remaining | running accuracy 0.562
  80/300 done | 88s elapsed | ~241s remaining | running accuracy 0.537
  96/300 done | 105s elapsed | ~223s remaining | running accuracy 0.552
  112/300 done | 123s elapsed | ~206s remaining | running accuracy 0.536
  128/300 done | 140s elapsed | ~188s remaining | running accuracy 0.562
  144/300 done | 155s elapsed | ~168s remaining | running accuracy 0.562
  160/300 done | 169s elapsed | ~148s remaining | running accuracy 0.581
  176/300 done | 184s elapsed | ~130s remaining | running accuracy 0.602
  192/300 done | 201s elapsed | ~113s remaining | running accuracy 0.604
  208/300 done | 219s elapsed | ~97s remaining | running accuracy 0.596
  224/300 done | 236s

## Headroom decision

Combines both models' results into the one number the decision rule is
based on: the paired accuracy gap and its confidence interval.

In [12]:
correct_teacher = [r["correct"] for r in teacher_summary["records"]]
correct_student = [r["correct"] for r in student_summary["records"]]

gap, (gap_low, gap_high) = paired_bootstrap_diff_ci(correct_teacher, correct_student)

print(f"Teacher ({TEACHER}): {teacher_summary['accuracy']:.1%} "
      f"(95% CI {teacher_summary['ci_95'][0]:.1%}-{teacher_summary['ci_95'][1]:.1%})")
print(f"Student ({STUDENT}): {student_summary['accuracy']:.1%} "
      f"(95% CI {student_summary['ci_95'][0]:.1%}-{student_summary['ci_95'][1]:.1%})")
print(f"\nGap (teacher - student): {gap:.1%} "
      f"(95% CI {gap_low:.1%} to {gap_high:.1%})")

# Decision rule, agreed as a team before running this notebook (see the
# intro cell). Edit the thresholds here if the team agrees on different ones.
if gap >= 0.10 and gap_low > 0:
    verdict = ("CLEAR HEADROOM: the teacher is meaningfully better on this task. "
               "Proceed with the distillation experiment as planned.")
elif gap < 0.05 or gap_high <= 0 <= gap_low or (gap_low <= 0 and gap_high >= 0):
    verdict = ("LITTLE TO NO HEADROOM: the teacher is not clearly better here. "
               "Distilling on GSM8K is unlikely to transfer much. Consider a "
               "harder dataset (e.g. MATH levels 1-3) or a different "
               "teacher/student pairing before running the main experiment.")
else:
    verdict = ("AMBIGUOUS: some headroom, but the gap is not clearly large. "
               "Proceeding is reasonable, but expect small effects and plan "
               "for more seeds in the main experiment.")

print(f"\nVERDICT: {verdict}")

Teacher (Qwen/Qwen3-4B): 83.0% (95% CI 78.3%-86.8%)
Student (Qwen/Qwen3-0.6B): 56.7% (95% CI 51.0%-62.2%)

Gap (teacher - student): 26.3% (95% CI 21.0% to 31.7%)

VERDICT: CLEAR HEADROOM: the teacher is meaningfully better on this task. Proceed with the distillation experiment as planned.


## Save results

Saves a summary JSON (for quick reference and for the team's results log)
and one CSV per model with every question, response, and grade (for
reading back specific wrong answers later).

In [13]:
out_dir = Path("/kaggle/working/headroom_results")
out_dir.mkdir(exist_ok=True)

config = {
    "teacher": TEACHER,
    "student": STUDENT,
    "n_eval": N_EVAL,
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "seed": SEED,
    "half_dtype": str(HALF),
    "gpu": torch.cuda.get_device_name(0),
}

summary_out = {
    "config": config,
    "teacher": {k: v for k, v in teacher_summary.items() if k != "records"},
    "student": {k: v for k, v in student_summary.items() if k != "records"},
    "gap": round(float(gap), 4),
    "gap_ci_95": (round(float(gap_low), 4), round(float(gap_high), 4)),
    "verdict": verdict,
}

summary_file = out_dir / "headroom_summary.json"
summary_file.write_text(json.dumps(summary_out, indent=2))
print(f"Saved summary to {summary_file}")

for name, summary in [("teacher", teacher_summary), ("student", student_summary)]:
    records_file = out_dir / f"headroom_records_{name}.csv"
    with open(records_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary["records"][0].keys()))
        writer.writeheader()
        writer.writerows(summary["records"])
    print(f"Saved {len(summary['records'])} records to {records_file}")

Saved summary to /kaggle/working/headroom_results/headroom_summary.json
Saved 300 records to /kaggle/working/headroom_results/headroom_records_teacher.csv
Saved 300 records to /kaggle/working/headroom_results/headroom_records_student.csv
